# Teaching Gemma to write Python

## Overview
| | dense | mixture-of-experts |
|---|---|---|
| FFN | one dense block, width 1536 | shared width 768 + 3 experts of width 384, top-2 |
| parameters, total | 13.05M | 15.43M |
| parameters used per token | 13.05M | 13.07M |
| validation loss | 1.0639 | 1.0637 |
| HumanEval+ mini, docstring | 15/164 | 16/164 |
| HumanEval+ mini, stripped | 19/164 | 18/164 |
| 4 epochs on an RTX 3090 | 7h57m | 12h27m |

Both arms use the same number of parameters per token, so they do the same
arithmetic per token; the MoE adds storage without adding compute. This notebook walks through the
data, the collator, the two architectures, and inference with the trained weights.

## Setup

Run this notebook from `examples/py-edu-gemma-moe`; the config, data, and output
paths are relative to it. The `.venv` needs torch, transformers, datasets, and
evalplus.

In [1]:
import os
from pathlib import Path

assert Path("configs/dense.yaml").exists(), "Run this notebook from examples/py-edu-gemma-moe"
os.environ["HF_DATASETS_DISABLE_PROGRESS_BARS"] = "1"  # keep the saved outputs clean

HF_REPO = "Blankyy/py-edu-gemma-15m"  # public tokenizer and weights

In [2]:
import statistics

import ignite.distributed as idist
import torch
import yaml
from huggingface_hub import hf_hub_download
from torch import nn
from transformers import PreTrainedTokenizerFast

from config import ProjectConfig
from utils import build_dataloaders, build_dataset, build_model, flatten_sweep_config, load_config

device = idist.device()
print("torch", torch.__version__, "device", device)

torch 2.15.0 device mps


## The two configs

Everything except the FFN block is identical between the two YAML files. That is the
whole experiment: one factor changed.

### `configs/dense.yaml`

In [3]:
dense_config = load_config("configs/dense.yaml", ProjectConfig)
moe_config = load_config("configs/moe.yaml", ProjectConfig)
print(yaml.safe_dump(dense_config.model_dump(by_alias=True), sort_keys=False, width=100).rstrip())

project_name: py-edu-gemma
preprocessor:
  _target_: preprocessors.python_bpe.load_python_bpe_tokenizer
  vocab_size: 8192
  cache_dir: outputs/tokenizers
model:
  _target_: models.gemma4_dense.Gemma4DenseModel
  collate_fn_target: models.gemma4_dense.CausalLMCollateFn
  hidden_size: 256
  num_layers: 8
  num_attention_heads: 4
  num_key_value_heads: 2
  dense_intermediate_size: 1536
  head_dim: 64
  layer_types: null
  layer_pattern: sssg
  sliding_window: 512
  rope_theta: 10000.0
  global_rope_theta: 1000000.0
  rotary_fraction: 1.0
  global_rotary_fraction: 0.25
  global_num_key_value_heads: 2
  global_head_dim: 64
  global_key_equals_value: true
  attention_dropout: 0.0
optimizer:
  _target_: torch.optim.AdamW
  lr: 0.0006
loss:
  _target_: models.gemma4_dense.CausalLMCrossEntropyLoss
  ignore_index: -100
data:
  dataset:
    _target_: dataset_impl.python_edu.load_function_dataset
    path: Avelina/python-edu-cleaned
    dataset_size: null
    split: train
    max_length: 512
    

### `configs/moe.yaml`

Same structure; only the FFN keys differ.

In [4]:
print(yaml.safe_dump(moe_config.model_dump(by_alias=True), sort_keys=False, width=100).rstrip())

project_name: py-edu-gemma
preprocessor:
  _target_: preprocessors.python_bpe.load_python_bpe_tokenizer
  vocab_size: 8192
  cache_dir: outputs/tokenizers
model:
  _target_: models.gemma4_moe.Gemma4MoEModel
  collate_fn_target: models.gemma4_moe.CausalLMCollateFn
  hidden_size: 256
  num_layers: 8
  num_attention_heads: 4
  num_key_value_heads: 2
  dense_intermediate_size: 768
  expert_dim: 384
  num_experts: 3
  top_k: 2
  head_dim: 64
  layer_types: null
  layer_pattern: sssg
  sliding_window: 512
  rope_theta: 10000.0
  global_rope_theta: 1000000.0
  rotary_fraction: 1.0
  global_rotary_fraction: 0.25
  global_num_key_value_heads: 2
  global_head_dim: 64
  global_key_equals_value: true
  attention_dropout: 0.0
optimizer:
  _target_: torch.optim.AdamW
  lr: 0.0006
loss:
  _target_: models.gemma4_moe.GemmaMoECausalLMLoss
  ignore_index: -100
  aux_loss_coef: 0.01
data:
  dataset:
    _target_: dataset_impl.python_edu.load_function_dataset
    path: Avelina/python-edu-cleaned
    datas

### The difference

Every key where the two files disagree, with no nesting to read past.

In [5]:
dense_flat = flatten_sweep_config(dense_config.model_dump(by_alias=True))
for key, value in flatten_sweep_config(moe_config.model_dump(by_alias=True)).items():
    previous = dense_flat.get(key, "<unset>")
    if value != previous:
        print(f"{key}: {previous!r} -> {value!r}")

model._target_: 'models.gemma4_dense.Gemma4DenseModel' -> 'models.gemma4_moe.Gemma4MoEModel'
model.collate_fn_target: 'models.gemma4_dense.CausalLMCollateFn' -> 'models.gemma4_moe.CausalLMCollateFn'
model.dense_intermediate_size: 1536 -> 768
model.expert_dim: '<unset>' -> 384
model.num_experts: '<unset>' -> 3
model.top_k: '<unset>' -> 2
loss._target_: 'models.gemma4_dense.CausalLMCrossEntropyLoss' -> 'models.gemma4_moe.GemmaMoECausalLMLoss'
loss.aux_loss_coef: '<unset>' -> 0.01
output.run_name: 'gemma4_dense__py_edu_15m' -> 'gemma4_moe__py_edu_15m'


## Tokenizer

A byte-level BPE with an 8192-token vocabulary, fitted once on the raw corpus and
published with the weights, so the notebook downloads it instead of refitting — a
refit has to read the full 6 GB corpus first. `<eos>` is trained in, `<pad>` is
appended, so the embedding has 8193 rows and `pad_token_id == 8192`.

In [6]:
tokenizer = PreTrainedTokenizerFast(
    tokenizer_file=hf_hub_download(repo_id=HF_REPO, filename="tokenizer.json"),
    eos_token="<eos>",
    pad_token="<pad>",
)
print("vocab size:", len(tokenizer), "(8192 BPE tokens + <pad>)")
print("eos:", tokenizer.eos_token_id, " pad:", tokenizer.pad_token_id)

example = "def add(a, b):\n    return a + b\n"
ids = tokenizer.encode(example, add_special_tokens=False)
print(len(ids), "tokens:", ids)
print(repr(tokenizer.decode(ids)))

vocab size: 8193 (8192 BPE tokens + <pad>)
eos: 0  pad: 8192
13 tokens: [324, 859, 8, 65, 12, 318, 295, 265, 344, 274, 340, 318, 199]
'def add(a, b):\n    return a + b\n'


## Dataset

The source is `Avelina/python-edu-cleaned`, a corpus of functions extracted from
public Python projects. The full run sees 3.35M training functions (52,270 batches
of 64 per epoch). Here we load 5,000 so the notebook stays interactive. A capped load
streams the corpus rather than downloading all 6 GB, so the first run pulls a few
megabytes and the extraction is cached by fingerprint after that.

In [7]:
WALKTHROUGH_SIZE = 5000
dense_config.data.dataset.dataset_size = WALKTHROUGH_SIZE

dataset = build_dataset(dense_config.data.dataset, dense_config.data.transform, tokenizer)
raw = dataset.dataset  # the (prompt, completion) pairs before tokenization
print(f"{len(dataset):,} functions loaded for this walkthrough")

5,000 functions loaded for this walkthrough


### A look at the data

Dataset contains raw Python files with functions extracted.
Each row is a prompt (imports, signature, and usually a docstring) and a completion (the function body).
The relaxed corpus used for both runs keeps bare functions too, so both shapes appear.

In [8]:
from datasets import load_dataset

def get_raw_file():
    data_config = dense_config.data.dataset
    original_dataset = load_dataset(data_config.path, split=data_config.split, streaming=True)
    first_text = None
    for row in original_dataset.take(WALKTHROUGH_SIZE):
        text = row["text"]
        if first_text is None:
            first_text = text
        if (
            '"""' in text
            and all(ord(c) < 128 for c in text)
            and len(tokenizer.encode(text)) < 1024
        ):
            return text
    return first_text

raw_file = get_raw_file()
print(f"raw file: {len(raw_file)} characters, {len(raw_file.splitlines())} lines")
print(raw_file)

raw file: 1710 characters, 53 lines
import shutil
import os


class changed_dir:  # pragma: nocover # noqa  -> disable=N801
    """
    Changes into a arbitrary directory, switching back to the directory after execution of the with statement.

    directory:
        the directory that should be changed into.
    create_if_not_exists:
        set this to False, if the chgdir should fail loudly. If set to `True`,
        tries to create the directory if it doesn't exist

    fallback_to_tempdir_as_last_resort:
        assumes create_if_not_exists is also True
        creates a tmp directory as last resort, and does work there. Defaults to False.
    """

    def __init__(self, directory, create_if_not_exists=True):
        self.old_dir = os.getcwd()
        self.new_dir = directory
        self.success = None
        self.create_if_not_exists = create_if_not_exists

    def __enter__(self):
        try:
            try:
                # see if we can enter the directory
                

Each row is a `prompt` (imports, signature, and usually a docstring) and a
`completion` (the function body). The relaxed corpus used for both runs keeps bare
functions too, so both shapes appear.

In [9]:
def first_matching(predicate):
    return next((i for i, row in enumerate(raw) if predicate(row)), 0)


def show(index):
    row = raw[index]
    print(f"--- index {index} ---")
    print("prompt:")
    print(row["prompt"])
    print("completion:")
    print(row["completion"])


documented = first_matching(lambda r: '"""' in r["prompt"] and all(ord(c) < 128 for c in r["prompt"] + r["completion"]))
bare = first_matching(
    lambda r: (
        '"""' not in r["prompt"]
        and len(r["prompt"].splitlines()) == 1
        and all(ord(c) < 128 for c in r["prompt"] + r["completion"])
    )
)
show(documented)
show(bare)

--- index 5 ---
prompt:
def print_melon(melon_info):
    """Print each melon with corresponding attribute information."""

completion:
    for name, attributes in melon_info.items():
    	seedless, price = attributes
    	print(name.upper(), seedless, price)
--- index 2 ---
prompt:
def fibo(n_int):

completion:
    if n_int == 0:
        return 0
    if n_int == 1:
        return 1
    if n_int >= 2:
        return (fibo(n_int-1)+fibo(n_int-2))       



Lengths are capped at 512 tokens by the loader: examples whose prompt plus body plus
`<eos>` would not fit are dropped at extraction time, before training starts.

In [10]:
lengths = sorted(
    len(tokenizer.encode(row["prompt"], add_special_tokens=False))
    + len(tokenizer.encode(row["completion"], add_special_tokens=False))
    + 1
    for row in raw
)


def percentile(fraction):
    return lengths[int(fraction * (len(lengths) - 1))]


print(
    f"tokens per example: mean {statistics.mean(lengths):.0f} · median {percentile(0.5)} · "
    f"p90 {percentile(0.9)} · p99 {percentile(0.99)} · max {lengths[-1]}"
)
print(f"{sum(length > 400 for length in lengths)} of {len(lengths)} are above 400 tokens")

tokens per example: mean 102 · median 74 · p90 218 · p99 444 · max 511
82 of 5000 are above 400 tokens


### What the model actually trains on

The transform builds `prompt_ids + body_ids + <eos>`, shifts it by one, and masks the
prompt with `-100` so the loss is only on the body. This is what stops the model from
learning to reproduce imports and docstrings it already has.

In [11]:
item = dataset[documented]
prompt_len = item.eval_input_ids.shape[0]
print("input_ids:", tuple(item.train_input_ids.shape), " labels:", tuple(item.train_label_ids.shape))
print("prompt tokens:", prompt_len)
print("masked labels:", int((item.train_label_ids == -100).sum()), "of", len(item.train_label_ids))


def preview(values, start=0, count=14):
    return " ".join(
        "·" if value == -100 else repr(tokenizer.decode([value])) for value in values[start : start + count].tolist()
    )


start = max(0, prompt_len - 4)
print("input :", preview(item.train_input_ids, start))
print("labels:", preview(item.train_label_ids, start))
print("first body token:", repr(tokenizer.decode([item.train_label_ids[prompt_len - 1].item()])))

input_ids: (57,)  labels: (57,)
prompt tokens: 23
masked labels: 22 of 57
input : ' attribute' ' information' '."""' '\n' '   ' ' for' ' name' ',' ' attributes' ' in' ' me' 'lon' '_' 'info'
labels: · · · '   ' ' for' ' name' ',' ' attributes' ' in' ' me' 'lon' '_' 'info' '.'
first body token: '   '


## Dataloaders

`build_dataloaders` takes the 5,000 functions and splits them 80/10/10 with a fixed
seed. `CausalLMCollateFn` left-pads each batch: because generation appends tokens on
the right, the last column has to be the last real token. Padding is masked in the
attention mask and ignored in the labels.

`num_workers` is forced to 0 because a notebook kernel cannot spawn the loader
workers the training config uses on a GPU box.

In [12]:
dense_config.data.dataloader.num_workers = 0
train_loader, val_loader, test_loader = build_dataloaders(
    dense_config.data,
    tokenizer,
    dense_config.seed,
    collate_fn_target=dense_config.model.collate_fn_target,
)
print(
    f"train {len(train_loader)} batches · val {len(val_loader)} · test {len(test_loader)} "
    f"(batch size {dense_config.data.dataloader.batch_size})"
)

torch.manual_seed(dense_config.seed)  # same batch order on every run
batch = next(iter(train_loader))
print({name: tuple(value.shape) for name, value in batch.items()})

train 63 batches · val 8 · test 8 (batch size 64)
{'input_ids': (64, 477), 'attention_mask': (64, 477), 'labels': (64, 477)}


In [13]:
for row in range(2):
    ids = batch["input_ids"][row].tolist()
    pad = tokenizer.pad_token_id
    left_pad = 0
    while left_pad < len(ids) and ids[left_pad] == pad:
        left_pad += 1
    row_labels = batch["labels"][row]
    first_scored = int((row_labels != -100).nonzero()[0])
    print(f"row {row}: {left_pad} pad tokens · {len(ids) - left_pad} real tokens · {len(ids)} columns")
    print(f"  first scored token at column {first_scored}: {tokenizer.decode([int(row_labels[first_scored])])!r}")
    print(f"  labels around it: {row_labels[max(0, first_scored - 2) : first_scored + 3].tolist()}")

row 0: 357 pad tokens · 120 real tokens · 477 columns
  first scored token at column 369: '   '
  labels around it: [-100, -100, 259, 1026, 262]
row 1: 319 pad tokens · 158 real tokens · 477 columns
  first scored token at column 419: '   '
  labels around it: [-100, -100, 259, 7144, 63]


## Models

The MoE block replaces the dense FFN with a shared FFN plus routed experts. The key
number is active parameters: 768 shared + 2 of 3 experts × 384 = 1536, exactly the
dense FFN width. So both models do the same work per token, and the extra 2.4M
parameters are storage only until the experts specialise.

In [14]:
dense_model = build_model(dense_config.model, device, vocab_size=len(tokenizer), pad_token_id=tokenizer.pad_token_id)
moe_model = build_model(moe_config.model, device, vocab_size=len(tokenizer), pad_token_id=tokenizer.pad_token_id)


def count_parameters(model):
    return sum(parameter.numel() for parameter in model.parameters())


moe_cfg = moe_config.model
unused_per_layer = (moe_cfg.num_experts - moe_cfg.top_k) * 3 * moe_cfg.expert_dim * moe_cfg.hidden_size
unused_experts = moe_cfg.num_layers * unused_per_layer
moe_active = count_parameters(moe_model) - unused_experts

dense_params = count_parameters(dense_model)
print(f"dense: {dense_params / 1e6:.2f}M total · {dense_params / 1e6:.2f}M per token")
print(f"moe:   {count_parameters(moe_model) / 1e6:.2f}M total · {moe_active / 1e6:.2f}M per token")

hidden = dense_config.model.hidden_size
dense_ffn = 3 * hidden * dense_config.model.dense_intermediate_size
moe_ffn = 3 * hidden * (moe_cfg.dense_intermediate_size + moe_cfg.top_k * moe_cfg.expert_dim)
print(f"FFN parameters per layer: dense {dense_ffn:,} · moe active {moe_ffn:,}")
assert dense_ffn == moe_ffn

dense: 13.05M total · 13.05M per token
moe:   15.43M total · 13.07M per token
FFN parameters per layer: dense 1,179,648 · moe active 1,179,648


## Trained checkpoints

Both runs were published to
[`Blankyy/py-edu-gemma-15m`](https://huggingface.co/Blankyy/py-edu-gemma-15m), so the
notebook needs no credentials. The two files here are the exact `best.pt` state dicts
with the optimizer stripped: float32, `{"model": state_dict}`, plain keys with no
`module.` prefix. They are ~52 MB each instead of 157–185 MB with the optimizer.

`hf_hub_download` caches under `~/.cache/huggingface`, so the second run is offline
and instant; deleting that cache is the only way to force a re-download.

In [15]:

def load_weights(model, filename):
    path = hf_hub_download(repo_id=HF_REPO, filename=filename)
    checkpoint = torch.load(path, map_location="cpu", weights_only=True)
    model.load_state_dict(checkpoint["model"])
    return model.to(device).eval()


load_weights(dense_model, "dense.weights.pt")
load_weights(moe_model, "moe.weights.pt")
print("weights loaded")

weights loaded


## Inference

The trainer's generation loop is greedy decoding with a frozen EOS. This is the same
loop without the Ignite/Accelerate machinery; batching needs left padding, which is
why the collator does it.

In [16]:
@torch.no_grad()
def generate_greedy(model, prompt, max_new_tokens=96):
    ids = torch.tensor([tokenizer.encode(prompt, add_special_tokens=False)], device=device)
    attention_mask = torch.ones_like(ids)
    generated = ids.clone()
    for _ in range(max_new_tokens):
        logits = model(generated, attention_mask=attention_mask)
        next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
        finished = generated[:, -1:].eq(tokenizer.eos_token_id)
        next_token = torch.where(finished, generated[:, -1:], next_token)
        generated = torch.cat([generated, next_token], dim=-1)
        attention_mask = torch.cat(
            [attention_mask, torch.ones((1, 1), dtype=attention_mask.dtype, device=device)], dim=-1
        )
        if generated[:, -1].eq(tokenizer.eos_token_id).all():
            break
    return tokenizer.decode(generated[0, ids.shape[1] :].tolist(), skip_special_tokens=True)

First, a held-out function from the validation split. The prompt is everything the
model is given; the completion below it is generated.

In [17]:
item = next(val_loader.dataset[i] for i in range(len(val_loader.dataset)) if '"""' in val_loader.dataset[i].source)
print(item.source)
for name, model in (("dense", dense_model), ("moe", moe_model)):
    print(f"\n=== {name} completion ===")
    print(item.source + generate_greedy(model, item.source, max_new_tokens=64))

def walk_together(*readers):
    """ Walk a set of readers and return lists of records from each
        reader, with None if no record present.  Caller must check the
        inputs are sorted in the same way and use the same reference
        otherwise behaviour is undefined.
    """


=== dense completion ===


def walk_together(*readers):
    """ Walk a set of readers and return lists of records from each
        reader, with None if no record present.  Caller must check the
        inputs are sorted in the same way and use the same reference
        otherwise behaviour is undefined.
    """
    if not readers:
        return []
    return [readers[0]] + walk_together(readers[1:])


=== moe completion ===


def walk_together(*readers):
    """ Walk a set of readers and return lists of records from each
        reader, with None if no record present.  Caller must check the
        inputs are sorted in the same way and use the same reference
        otherwise behaviour is undefined.
    """
    outer_list = []
    for readers in readers:
        outer_list.append(readers[0])
    return outer_list



### HumanEval

The evaluation prompts are the same shape: imports, signature, docstring. `evalplus`
ships the 164 problems, and `_strip_docstring` is the exact helper the trainer used
for the `nodoc` numbers above.

These are 15M-parameter models trained for four epochs, so most single samples are
wrong. That is why the score is 15–16 of 164 rather than 150.

In [18]:
from evalplus.data import get_human_eval_plus
from trainer import _strip_docstring

problems = get_human_eval_plus()
problem = problems["HumanEval/0"]
print(problem["prompt"])
print("--- stripped ---")
print(_strip_docstring(problem["prompt"]))

from typing import List


def has_close_elements(numbers: List[float], threshold: float) -> bool:
    """ Check if in given list of numbers, are any two numbers closer to each other than
    given threshold.
    >>> has_close_elements([1.0, 2.0, 3.0], 0.5)
    False
    >>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)
    True
    """

--- stripped ---
from typing import List


def has_close_elements(numbers: List[float], threshold: float) -> bool:



In [19]:
for name, model in (("dense", dense_model), ("moe", moe_model)):
    for label, prompt in (("with docstring", problem["prompt"]), ("stripped", _strip_docstring(problem["prompt"]))):
        print(f"\n=== {name} · {label} ===")
        print(generate_greedy(model, prompt, max_new_tokens=96))


=== dense · with docstring ===


    for i in range(len(numbers)):
        for j in range(i + 1, len(numbers)):
            if numbers[i] + numbers[j] == threshold:
                return True
    return False


=== dense · stripped ===
    return len(set(numbers)) == len(numbers)


=== moe · with docstring ===


    for i in range(len(numbers)):
        if numbers[i] > threshold:
            return True
    return False


=== moe · stripped ===


    return any(abs(x - threshold) < 1 for x in numbers)



### Loss on the same data

The whole validation split of the 5,000-example subset, scored with plain
cross-entropy for both models. This is 500 held-out functions; the full-corpus test
split behind the numbers at the end has ~418k examples. At n=500 the two models land
within ~0.01 of each other and the ordering flips with the sample, which is exactly
the noise this notebook is about. Don't quote this number.

In [20]:
criterion = nn.CrossEntropyLoss(ignore_index=-100)
totals = {"dense": 0.0, "moe": 0.0}
batches = 0

for batch in val_loader:
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["labels"].to(device)
    batches += 1
    for name, model in (("dense", dense_model), ("moe", moe_model)):
        with torch.no_grad():
            logits = model(input_ids, attention_mask=attention_mask)
            totals[name] += criterion(logits.reshape(-1, logits.shape[-1]), labels.reshape(-1)).item()

print(f"mean cross-entropy over {batches} validation batches:")
for name, total in totals.items():
    print(f"  {name:5s} {total / batches:.4f}")

mean cross-entropy over 8 validation batches:
  dense 1.0613
  moe   1.0699


### Router behaviour (MoE only)

If the experts had specialised, tokens would concentrate on one expert. Instead every
layer sits at one third per expert, which is what the 0.01 load-balancing term asks
for. That is consistent with the identical loss: at this size the extra experts are
not buying anything.

In [21]:
from models.gemma4_moe import load_balancing_loss

with torch.no_grad():
    _, router_info = moe_model(input_ids, attention_mask=attention_mask, return_router_info=True)

usage = router_info[:, 1, :].cpu()
print("expert share of routed tokens, per layer (top-2 of 3):")
for layer, row in enumerate(usage):
    print(f"  layer {layer}: " + "  ".join(f"{value:.3f}" for value in row.tolist()))
print(f"\nuniform would be {1 / moe_cfg.num_experts:.3f} per expert")
print(f"load-balancing loss {load_balancing_loss(router_info).item():.4f} (1.0 = perfectly uniform)")

expert share of routed tokens, per layer (top-2 of 3):
  layer 0: 0.319  0.348  0.333
  layer 1: 0.319  0.316  0.365
  layer 2: 0.332  0.352  0.316
  layer 3: 0.305  0.335  0.360
  layer 4: 0.324  0.322  0.354
  layer 5: 0.346  0.317  0.336
  layer 6: 0.344  0.303  0.353
  layer 7: 0.305  0.345  0.350

uniform would be 0.333 per expert
load-balancing loss 1.0016 (1.0 = perfectly uniform)


## What the full runs measured

These are the numbers from the training logs, not from this 5,000-example subset.

| | dense | MoE |
|---|---|---|
| parameters, total | 13,051,400 | 15,425,056 |
| parameters per token | 13,051,400 | 13,065,760 |
| validation loss | 1.0639 | 1.0637 |
| token accuracy | 0.7494 | 0.7495 |
| HumanEval+ base, docstring | 15/164 | 16/164 |
| HumanEval+ base, stripped | 19/164 | 18/164 |
| HumanEval+ plus, docstring | 12/164 | 13/164 |
| HumanEval+ plus, stripped | 16/164 | 15/164 |
| training time, 4 epochs | 7h57m | 12h27m |
| seconds per step | 0.1380 | 0.2144 |

In a pretraining-only budget, the extra 2.4M expert parameters moved the loss by
0.0002 and HumanEval by one task, at 1.6× the training cost. Post-training could
still make the experts pay off; this run does not show that they do on their own.

To reproduce a run, `uv run python main.py configs/dense.yaml` (or `configs/moe.yaml`).
See the README for cloud launching.

The weights behind these numbers are published at
[`Blankyy/py-edu-gemma-15m`](https://huggingface.co/Blankyy/py-edu-gemma-15m), along
with the full training checkpoints.